In [1]:
import numpy as np
import pandas as pd

In [36]:
# loading data
books = pd.read_csv("Books.csv", low_memory=False)
ratings = pd.read_csv("Ratings.csv", low_memory=False)
users = pd.read_csv("Users.csv", low_memory=False)

# studying data

In [43]:
books.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  object
 1   Book-Title           271360 non-null  object
 2   Book-Author          271358 non-null  object
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  object
 5   Image-URL-S          271360 non-null  object
 6   Image-URL-M          271360 non-null  object
 7   Image-URL-L          271357 non-null  object
dtypes: object(8)
memory usage: 16.6+ MB


In [44]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1149780 entries, 0 to 1149779
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1149780 non-null  int64 
 1   ISBN         1149780 non-null  object
 2   Book-Rating  1149780 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 26.3+ MB


In [45]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278858 entries, 0 to 278857
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   User-ID   278858 non-null  int64  
 1   Location  278858 non-null  object 
 2   Age       168096 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 6.4+ MB


In [46]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271360, 8)
(1149780, 3)
(278858, 3)


In [18]:
ratings.isnull().sum()

User-ID        0
ISBN           0
Book-Rating    0
dtype: int64

In [17]:
users.isnull().sum()

User-ID          0
Location         0
Age         110762
dtype: int64

In [19]:
books.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64

In [49]:
# users.duplicated().sum()
# books.duplicated().sum()
ratings.duplicated().sum()

np.int64(0)

In [50]:
books = books[[
    'ISBN',
    'Book-Title',
    'Book-Author',
    'Year-Of-Publication',
    'Publisher',
    'Image-URL-L'
]]

In [51]:
# merging book and rating columns
book_ratings = ratings.merge(books, on='ISBN')
book_ratings.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,http://images.amazon.com/images/P/0155061224.0...
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,http://images.amazon.com/images/P/0446520802.0...
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,http://images.amazon.com/images/P/052165615X.0...
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,http://images.amazon.com/images/P/0521795028.0...


In [24]:
# removing dublicates
book_ratings = book_ratings.drop_duplicates()

In [52]:
# counting number of ratings and removing book with less than 50 ratings
num_ratings = book_ratings.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_ratings.rename(columns={'Book-Rating':'num_ratings'}, inplace=True)
popular_books = num_ratings[num_ratings['num_ratings'] >= 50]

In [53]:
avg_rating = book_ratings.groupby('Book-Title')['Book-Rating'].mean().reset_index()
avg_rating.rename(columns={'Book-Rating':'avg_rating'}, inplace=True)

In [29]:
popular_df = avg_rating.merge(popular_books, on='Book-Title')

In [30]:
final_books = book_ratings[[
    'ISBN',
    'Book-Title',
    'Book-Author',
    'Publisher',
    'Image-URL-L'
]]

final_books = final_books.drop_duplicates()

In [33]:
final_books.to_csv("final_books.csv", index=False)
final_books.head(1)

,ISBN,Book-Title,Book-Author,Publisher,Image-URL-L
0,034545104X,Flesh Tones: A Novel,M. J. Rose,Ballantine Books,http://images.amazon.com/images/P/034545104X.0...


# Exploratory Data Analysis (EDA)

In [54]:
print("Total Books:", books['Book-Title'].nunique())

Total Books: 242135


In [55]:
print("Total Users:", ratings['User-ID'].nunique())

Total Users: 105283


In [56]:
print("Total Ratings:", ratings.shape[0])

Total Ratings: 1149780


In [57]:
top_books = book_ratings.groupby('Book-Title')['Book-Rating'].count() \
                        .sort_values(ascending=False) \
                        .head(10)

top_books

Book-Title
Wild Animus                                        2502
The Lovely Bones: A Novel                          1295
The Da Vinci Code                                   898
A Painted House                                     838
The Nanny Diaries: A Novel                          828
Bridget Jones's Diary                               815
The Secret Life of Bees                             774
Divine Secrets of the Ya-Ya Sisterhood: A Novel     740
The Red Tent (Bestselling Backlist)                 723
Angels &amp; Demons                                 670
Name: Book-Rating, dtype: int64

In [58]:
book_stats = book_ratings.groupby('Book-Title').agg(
    avg_rating=('Book-Rating','mean'),
    total_ratings=('Book-Rating','count')
)

book_stats = book_stats[book_stats['total_ratings']>=50]

book_stats.sort_values('avg_rating',ascending=False).head(10)

,avg_rating,total_ratings
Book-Title,,
Free,8.017857,56
The Stand (The Complete and Uncut Edition),6.175439,57
Griffin &amp; Sabine: An Extraordinary Correspondence,6.041667,72
Harry Potter and the Prisoner of Azkaban (Book 3),5.852804,428
Harry Potter and the Goblet of Fire (Book 4),5.824289,387
The Little Prince,5.815603,141
The Cat in the Hat,5.754717,53
Harry Potter and the Sorcerer's Stone (Book 1),5.737410,278
The Hobbit,5.700000,80


# Popularity based Recommendation System

In [60]:
average_rating = book_ratings.groupby('Book-Title')['Book-Rating'].mean().reset_index()
average_rating.rename(columns={'Book-Rating':'Average-Rating'}, inplace=True)

In [61]:
num_rating = book_ratings.groupby('Book-Title')['Book-Rating'].count().reset_index()
num_rating.rename(columns={'Book-Rating':'Num-Ratings'}, inplace=True)

In [62]:
popular_df = average_rating.merge(num_rating,on='Book-Title')

In [63]:
popular_df = popular_df[popular_df['Num-Ratings']>=250]

In [64]:
popular_df = popular_df.sort_values(
    'Average-Rating',
    ascending=False
)

In [65]:
popular_df = popular_df.merge(
    books,
    on='Book-Title'
)

In [66]:
popular_df = popular_df.drop_duplicates('Book-Title')

In [67]:
popular_df = popular_df[[
    'Book-Title',
    'Book-Author',
    'Image-URL-L',
    'Publisher',
    'Average-Rating',
    'Num-Ratings'
]]

In [68]:
popular_df.head(1)

,Book-Title,Book-Author,Image-URL-L,Publisher,Average-Rating,Num-Ratings
0,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,http://images.amazon.com/images/P/0439136350.0...,Scholastic,5.852804,428
